# Mini-system example to inspect optimization constraints

This notebook creates a small two-region energy system with three time steps.

The system contains:

- two electricity sources,
- electricity demand in both regions,
- an electricity transmission connection between the regions.

After constructing and optimizing the system, the Pyomo variables, constraints,
and objective function generated by FINE are printed.

In [ ]:
import fine as fn
import pandas as pd
import pyomo.environ as pyo

In [ ]:
# Define the mini-system input data
locations = {"Region_A", "Region_B"}
commodities = {"electricity"}

commodity_unit_dict = {
    "electricity": "MW",
}

number_of_time_steps = 3
hours_per_time_step = 1

In [ ]:
# Create the FINE energy system model
esM = fn.EnergySystemModel(
    locations=locations,
    commodities=commodities,
    numberOfTimeSteps=number_of_time_steps,
    commodityUnitsDict=commodity_unit_dict,
    hoursPerTimeStep=hours_per_time_step,
    costUnit="Euro",
    lengthUnit="km",
    verboseLogLevel=0,
)

In [ ]:
# Define the time series
time_steps = range(number_of_time_steps)

source_a_availability = pd.DataFrame(
    {
        "Region_A": [1.0, 1.0, 1.0],
        "Region_B": [0.0, 0.0, 0.0],
    },
    index=time_steps,
)

source_b_availability = pd.DataFrame(
    {
        "Region_A": [0.0, 0.0, 0.0],
        "Region_B": [1.0, 1.0, 1.0],
    },
    index=time_steps,
)

electricity_demand = pd.DataFrame(
    {
        "Region_A": [2.0, 3.0, 2.0],
        "Region_B": [4.0, 5.0, 6.0],
    },
    index=time_steps,
)

electricity_demand

,Region_A,Region_B
0,2.0,4.0
1,3.0,5.0
2,2.0,6.0


In [ ]:
# Add the electricity source in Region A
region_a_eligibility = pd.Series(
    {
        "Region_A": 1,
        "Region_B": 0,
    }
)

source_a_capacity_max = pd.Series(
    {
        "Region_A": 20.0,
        "Region_B": 0.0,
    }
)

esM.add(
    fn.Source(
        esM=esM,
        name="Source_A",
        commodity="electricity",
        hasCapacityVariable=True,
        locationalEligibility=region_a_eligibility,
        operationRateMax=source_a_availability,
        capacityMax=source_a_capacity_max,
        investPerCapacity=5,
        opexPerOperation=1,
        interestRate=0.08,
        economicLifetime=20,
    )
)

In [ ]:
# Add the electricity source in Region B
region_b_eligibility = pd.Series(
    {
        "Region_A": 0,
        "Region_B": 1,
    }
)

source_b_capacity_max = pd.Series(
    {
        "Region_A": 0.0,
        "Region_B": 20.0,
    }
)

esM.add(
    fn.Source(
        esM=esM,
        name="Source_B",
        commodity="electricity",
        hasCapacityVariable=True,
        locationalEligibility=region_b_eligibility,
        operationRateMax=source_b_availability,
        capacityMax=source_b_capacity_max,
        investPerCapacity=5,
        opexPerOperation=10,
        interestRate=0.08,
        economicLifetime=20,
    )
)

In [ ]:
# Add the electricity demand
esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity_demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=electricity_demand,
    )
)

In [ ]:
# Define the transmission connection
transmission_eligibility = pd.DataFrame(
    [
        [0, 1],
        [1, 0],
    ],
    index=["Region_A", "Region_B"],
    columns=["Region_A", "Region_B"],
)

transmission_distances = pd.DataFrame(
    [
        [0, 10],
        [10, 0],
    ],
    index=["Region_A", "Region_B"],
    columns=["Region_A", "Region_B"],
)

transmission_eligibility

,Region_A,Region_B
Region_A,0,1
Region_B,1,0


In [ ]:
# Add the transmission component
esM.add(
    fn.Transmission(
        esM=esM,
        name="Electricity_grid",
        commodity="electricity",
        hasCapacityVariable=True,
        locationalEligibility=transmission_eligibility,
        distances=transmission_distances,
        capacityMax=10,
        losses=0.01,
        investPerCapacity=1,
        opexPerOperation=0,
        interestRate=0.08,
        economicLifetime=40,
    )
)

In [ ]:
# Show the components
esM.componentNames

{'Source_A': 'SourceSinkModel',
 'Source_B': 'SourceSinkModel',
 'Electricity_demand': 'SourceSinkModel',
 'Electricity_grid': 'TransmissionModel'}

In [ ]:
# Optimize the system
esM.optimize(
    timeSeriesAggregation=False,
    solver=fn.utils.ImplementedSolvers.STANDARD_SOLVER.value,
)

Set parameter Threads to value 3
Set parameter LogFile to value ""
Set parameter QCPDual to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Rocky Linux 9.5 (Blue Onyx)")

CPU model: Intel(R) Xeon(R) Gold 6144 CPU @ 3.50GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 16 physical cores, 32 logical processors, using up to 3 threads

Non-default parameters:
TSPort  41955
QCPDual  1
Threads  3

Optimize a model with 32 rows, 34 columns and 82 nonzeros (Min)
Model fingerprint: 0x30da8b1e
Model has 10 linear objective coefficients
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [4e-01, 3e+04]
  Bounds range     [2e+00, 2e+01]
  RHS range        [0e+00, 0e+00]

Presolve removed 17 rows and 19 columns
Presolve time: 0.00s
Presolved: 15 rows, 15 columns, 39 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   1.415000e+01   0.000000e+00      0s
       5    6.9116671e+04   0.000000e+00   0

In [ ]:
# Check that the Pyomo model exists
pyM = esM.pyM

print(type(pyM))

<class 'pyomo.core.base.PyomoModel.ConcreteModel'>


In [ ]:
# Print all Pyomo variables
print("PYOMO VARIABLES")
print("=" * 80)

for variable in pyM.component_objects(pyo.Var, active=True):
    print(f"\nVariable group: {variable.name}")
    print("-" * 80)

    for index in variable:
        variable_entry = variable[index]

        print(
            f"{variable.name}[{index}]"
            f" = {pyo.value(variable_entry)}"
            f", lower bound = {variable_entry.lb}"
            f", upper bound = {variable_entry.ub}"
        )

PYOMO VARIABLES

Variable group: cap_srcSnk
--------------------------------------------------------------------------------
cap_srcSnk[('Region_A', 'Source_A', 0)] = 8.666666666666668, lower bound = 0, upper bound = 20.0
cap_srcSnk[('Region_B', 'Source_B', 0)] = 0.0, lower bound = 0, upper bound = 20.0

Variable group: nbReal_srcSnk
--------------------------------------------------------------------------------
nbReal_srcSnk[('Region_A', 'Source_A', 0)] = 8.666666666666668, lower bound = 0, upper bound = None
nbReal_srcSnk[('Region_B', 'Source_B', 0)] = 0.0, lower bound = 0, upper bound = None

Variable group: nbInt_srcSnk
--------------------------------------------------------------------------------

Variable group: commisBin_srcSnk
--------------------------------------------------------------------------------

Variable group: op_srcSnk
--------------------------------------------------------------------------------
op_srcSnk[('Region_A', 'Source_A', 0, 0, 0)] = 6.44444444444444

In [ ]:
# Collect all active Pyomo constraints in a table
constraint_records = []

for constraint_component in pyM.component_objects(
    pyo.Constraint,
    active=True,
):
    for index in constraint_component:
        constraint = constraint_component[index]

        if not constraint.active:
            continue

        constraint_records.append(
            {
                "constraint_group": constraint_component.name,
                "index": str(index),
                "expression": str(constraint.expr),
                "lower_bound": (
                    None
                    if constraint.lower is None
                    else str(constraint.lower)
                ),
                "body": str(constraint.body),
                "upper_bound": (
                    None
                    if constraint.upper is None
                    else str(constraint.upper)
                ),
            }
        )

constraints_df = pd.DataFrame(constraint_records)

print(
    f"Number of active constraint groups: "
    f"{constraints_df['constraint_group'].nunique()}"
)
print(f"Total number of active constraints: {len(constraints_df)}")

constraints_df

Number of active constraint groups: 10
Total number of active constraints: 32


,constraint_group,index,expression,lower_bound,body,upper_bound
0,ConstrCapToNbReal_srcSnk,"('Region_A', 'Source_A', 0)","cap_srcSnk[Region_A,Source_A,0] == nbReal_sr...",0.0,"cap_srcSnk[Region_A,Source_A,0] - nbReal_srcSn...",0.0
1,ConstrCapToNbReal_srcSnk,"('Region_B', 'Source_B', 0)","cap_srcSnk[Region_B,Source_B,0] == nbReal_sr...",0.0,"cap_srcSnk[Region_B,Source_B,0] - nbReal_srcSn...",0.0
2,DecommConstrCapacityDevelopment_srcSnk,"('Region_A', 'Source_A', 0)","decommis_srcSnk[Region_A,Source_A,0] == 0",0.0,"decommis_srcSnk[Region_A,Source_A,0]",0.0
3,DecommConstrCapacityDevelopment_srcSnk,"('Region_B', 'Source_B', 0)","decommis_srcSnk[Region_B,Source_B,0] == 0",0.0,"decommis_srcSnk[Region_B,Source_B,0]",0.0
4,InitialYear_srcSnk,"('Region_A', 'Source_A')","cap_srcSnk[Region_A,Source_A,0] == commis_sr...",0.0,"cap_srcSnk[Region_A,Source_A,0] - (commis_srcS...",0.0
5,InitialYear_srcSnk,"('Region_B', 'Source_B')","cap_srcSnk[Region_B,Source_B,0] == commis_sr...",0.0,"cap_srcSnk[Region_B,Source_B,0] - (commis_srcS...",0.0
6,ConstrOperation3_srcSnk,"('Region_A', 'Source_A', 0, 0, 0)","op_srcSnk[Region_A,Source_A,0,0,0] <= cap_sr...",None,"op_srcSnk[Region_A,Source_A,0,0,0] - cap_srcSn...",0.0
7,ConstrOperation3_srcSnk,"('Region_A', 'Source_A', 0, 0, 1)","op_srcSnk[Region_A,Source_A,0,0,1] <= cap_sr...",None,"op_srcSnk[Region_A,Source_A,0,0,1] - cap_srcSn...",0.0
8,ConstrOperation3_srcSnk,"('Region_A', 'Source_A', 0, 0, 2)","op_srcSnk[Region_A,Source_A,0,0,2] <= cap_sr...",None,"op_srcSnk[Region_A,Source_A,0,0,2] - cap_srcSn...",0.0
9,ConstrOperation3_srcSnk,"('Region_B', 'Source_B', 0, 0, 0)","op_srcSnk[Region_B,Source_B,0,0,0] <= cap_sr...",None,"op_srcSnk[Region_B,Source_B,0,0,0] - cap_srcSn...",0.0


In [ ]:
# List the generated constraint groups and their sizes
constraint_group_summary = (
    constraints_df.groupby("constraint_group")
    .size()
    .rename("number_of_constraints")
    .reset_index()
)

constraint_group_summary

,constraint_group,number_of_constraints
0,ConstrCapToNbReal_srcSnk,2
1,ConstrCapToNbReal_trans,2
2,ConstrOperation3_srcSnk,6
3,ConstrOperation_trans,6
4,ConstrSymmetricalCapacity_trans,2
5,DecommConstrCapacityDevelopment_srcSnk,2
6,DecommConstrCapacityDevelopment_trans,2
7,InitialYear_srcSnk,2
8,InitialYear_trans,2
9,commodityBalanceConstraint,6


In [ ]:
# Print the objective function
print("OBJECTIVE FUNCTION")
print("=" * 80)

for objective in pyM.component_objects(pyo.Objective, active=True):
    print(f"\nName: {objective.name}")
    print(f"Sense: {objective.sense}")
    print(f"Expression:\n{objective.expr}")
    print(f"\nOptimal value: {pyo.value(objective)}")

OBJECTIVE FUNCTION

Name: Obj
Sense: minimize
Expression:
(0.0*op_srcSnk[Region_B,Electricity_demand,0,0,0] + 0.0*op_srcSnk[Region_B,Electricity_demand,0,0,1] + 0.0*op_srcSnk[Region_B,Electricity_demand,0,0,2])/0.00034246575342465754*0.9259259259259267*1.08 + (0.0*op_srcSnk[Region_A,Electricity_demand,0,0,0] + 0.0*op_srcSnk[Region_A,Electricity_demand,0,0,1] + 0.0*op_srcSnk[Region_A,Electricity_demand,0,0,2])/0.00034246575342465754*0.9259259259259267*1.08 + (op_srcSnk[Region_A,Source_A,0,0,0] + op_srcSnk[Region_A,Source_A,0,0,1] + op_srcSnk[Region_A,Source_A,0,0,2])/0.00034246575342465754*0.9259259259259267*1.08 + (10.0*op_srcSnk[Region_B,Source_B,0,0,0] + 10.0*op_srcSnk[Region_B,Source_B,0,0,1] + 10.0*op_srcSnk[Region_B,Source_B,0,0,2])/0.00034246575342465754*0.9259259259259267*1.08 + (0.5092610441157533*commis_srcSnk[Region_A,Source_A,0] + 0.5092610441157533*commis_srcSnk[Region_B,Source_B,0] + 0.0*commis_srcSnk[Region_A,Source_A,0] + 0.0*commis_srcSnk[Region_B,Source_B,0]) + (0.0*op

In [ ]:
# Display optimization summaries
source_summary = esM.getOptimizationSummary(
    "SourceSinkModel",
    outputLevel=2,
)

source_summary

Region_A Region_B
Component          Property         Unit                           
Electricity_demand operation        [MW*h]             7.0     15.0
                   operation_annual [MW*h/a]       20440.0  43800.0
Source_A           NPVcontribution  [Euro]    69111.080262        0
                   TAC              [Euro/a]  69111.080262        0
                   capacity         [MW]          8.666667      NaN
                   capexCap         [Euro/a]      4.413596      NaN
                   commissioning    [MW]          8.666667      NaN
                   invest           [Euro]       43.333333      NaN
                   operation        [MW*h]       23.666667      NaN
                   operation_annual [MW*h/a]  69106.666667      NaN
                   opexOp           [Euro/a]  69106.666667      NaN

In [ ]:
# Display optimization summaries
transmission_summary = esM.getOptimizationSummary(
    "TransmissionModel",
    outputLevel=2,
)

transmission_summary

Region_A      Region_B
Component        Property         Unit     locationIn                         
Electricity_grid NPVcontribution  [Euro]   Region_A            0      2.795339
                                           Region_B     2.795339             0
                 TAC              [Euro/a] Region_A            0      2.795339
                                           Region_B     2.795339             0
                 capacity         [MW]     Region_A          NaN      6.666667
                                           Region_B     6.666667           NaN
                 capexCap         [Euro/a] Region_A          NaN      2.795339
                                           Region_B     2.795339           NaN
                 commissioning    [MW]     Region_A          NaN      6.666667
                                           Region_B     6.666667           NaN
                 invest           [Euro]   Region_A          NaN     33.333333
                                           Region_B    33.333333           NaN
                 operation        [MW*h]   Region_A          NaN     16.666667
                 operation_annual [MW*h/a] Region_A          NaN  48666.666667

In [ ]:
# Print every active constraint in a readable form
for number, row in constraints_df.iterrows():
    print(f"Constraint {number + 1}")
    print(f"Group: {row['constraint_group']}")
    print(f"Index: {row['index']}")
    print(f"Expression: {row['expression']}")
    print()

Constraint 1
Group: ConstrCapToNbReal_srcSnk
Index: ('Region_A', 'Source_A', 0)
Expression: cap_srcSnk[Region_A,Source_A,0]  ==  nbReal_srcSnk[Region_A,Source_A,0]

Constraint 2
Group: ConstrCapToNbReal_srcSnk
Index: ('Region_B', 'Source_B', 0)
Expression: cap_srcSnk[Region_B,Source_B,0]  ==  nbReal_srcSnk[Region_B,Source_B,0]

Constraint 3
Group: DecommConstrCapacityDevelopment_srcSnk
Index: ('Region_A', 'Source_A', 0)
Expression: decommis_srcSnk[Region_A,Source_A,0]  ==  0

Constraint 4
Group: DecommConstrCapacityDevelopment_srcSnk
Index: ('Region_B', 'Source_B', 0)
Expression: decommis_srcSnk[Region_B,Source_B,0]  ==  0

Constraint 5
Group: InitialYear_srcSnk
Index: ('Region_A', 'Source_A')
Expression: cap_srcSnk[Region_A,Source_A,0]  ==  commis_srcSnk[Region_A,Source_A,0] - decommis_srcSnk[Region_A,Source_A,0]

Constraint 6
Group: InitialYear_srcSnk
Index: ('Region_B', 'Source_B')
Expression: cap_srcSnk[Region_B,Source_B,0]  ==  commis_srcSnk[Region_B,Source_B,0] - decommis_srcSnk[

In [ ]:
try:
    from pyomo.contrib.latex_printer import latex_printer

    latex_printer_available = True
    print("Pyomo LaTeX printer is available.")
except ImportError:
    latex_printer_available = False
    print(
        "Pyomo LaTeX printer is not available in this "
        "Pyomo installation."
    )

Pyomo LaTeX printer is available.


In [ ]:
# Convert all active constraints to LaTeX when supported
latex_constraint_records = []

if latex_printer_available:
    for constraint_component in pyM.component_objects(
        pyo.Constraint,
        active=True,
    ):
        for index in constraint_component:
            constraint = constraint_component[index]

            if not constraint.active:
                continue

            latex_constraint_records.append(
                {
                    "constraint_group": constraint_component.name,
                    "index": str(index),
                    "latex_expression": latex_printer(
                        constraint.expr
                    ),
                }
            )

    latex_constraints_df = pd.DataFrame(
        latex_constraint_records
    )

    latex_constraints_df

## Interpretation

FINE creates the Pyomo optimization model automatically from the added
components.

The constraint groups above include:

- links between installed capacity and capacity variables,
- limits on source operation,
- fixed sink operation representing electricity demand,
- transmission capacity and operation constraints,
- commodity balance constraints for every region and time step.

The complete model contains 32 active constraints for this two-region,
three-time-step example.